In [1]:
!pip install tensorflow transformers tf-keras
!pip3 install newspaper3k
!pip install lxml[html_clean]
!pip install ipywidgets --upgrade
!pip install tweepy
!pip install requests
!pip install requests_oauthlib


[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import requests
from datetime import datetime, timedelta
from newspaper import Article
import nltk

nltk.download('punkt_tab')

# Calculate the start and end dates for this week
end_date = datetime.now()
start_date = end_date - timedelta(days=end_date.weekday() + 1)  # Monday of this week

base_url = "https://api.gdeltproject.org/api/v2/doc/doc"
params = {
    "query": "domain:motorsport.com (Formula One OR F1)",
    "mode": "ArtList",
    "maxrecords": 250,
    "startdatetime": start_date.strftime("%Y%m%d000000"),  # Format: YYYYMMDDHHMMSS
    "enddatetime": end_date.strftime("%Y%m%d235959"),      # Format: YYYYMMDDHHMMSS
    "format": "json"
}
# Make the request
response = requests.get(base_url, params=params)

# Check if the request was successful
if response.status_code == 200:
    print("Status Code:", response.status_code)
    articles = response.json()
    # print(articles) 

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\laure\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Status Code: 200


In [3]:
f1Articles = []
if 'articles' in articles:
    for article in articles['articles']:
        url = article.get('url', '')
        if "motorsport.com/f1/news" in url:
            f1Articles.append(article)
print(f1Articles)

[{'url': 'https://www.motorsport.com/f1/news/exclusive-inside-brazils-seven-year-search-for-its-next-f1-hero/10669897/', 'url_mobile': '', 'title': 'Inside Brazil seven - year search for its next F1 hero', 'seendate': '20241104T211500Z', 'socialimage': 'https://cdn-5.motorsport.com/images/amp/YKED4MO0/s6/ayrton-senna.jpg', 'domain': 'motorsport.com', 'language': 'English', 'sourcecountry': ''}, {'url': 'https://www.motorsport.com/f1/news/qatar-poised-for-buy-in-of-audis-f1-team/10671936/', 'url_mobile': '', 'title': 'Qatar poised for buy - in of Audi F1 team', 'seendate': '20241110T160000Z', 'socialimage': 'https://cdn-3.motorsport.com/images/amp/6l9BoZD0/s6/the-new-audi-sport-f1-concept-.jpg', 'domain': 'motorsport.com', 'language': 'English', 'sourcecountry': ''}, {'url': 'https://www.motorsport.com/f1/news/explained-the-brazilian-gp-red-flag-delay-that-red-bull-claims-cost-verstappen/10670213/', 'url_mobile': '', 'title': 'Explained : The Brazilian GP red flag delay that Red Bull cl

In [4]:
f1_articles_summaries = []
for article in f1Articles:
    url = article.get('url', '')
    try:
        currentArticle = Article(url)
        currentArticle.download()
        currentArticle.parse()
        f1_articles_summaries.append(currentArticle.text)
    except:
        print("Issue handling article")
# print(f1_articles_summaries)

In [5]:
from transformers import BartForConditionalGeneration, BartTokenizer

model_name = "facebook/bart-large-cnn"
model = BartForConditionalGeneration.from_pretrained(model_name)
tokenizer = BartTokenizer.from_pretrained(model_name)

def summarize_to_bullets(text):
    inputs = tokenizer(text, max_length=1024, return_tensors="pt", truncation=True)
    summary_ids = model.generate(inputs["input_ids"], max_length=50, min_length=10, length_penalty=2.0, num_beams=4, early_stopping=True)
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

summaryResult = []

for text in f1_articles_summaries:
    summary = summarize_to_bullets(text)
    summaryResult.append(summary)
print(summaryResult)

["Brazilians flocked to their home grand prix this weekend to cheer on Williams sensation Franco Colapinto. The country's last full-time F1 driver Felipe Massa is yet to find a successor after his retirement at the", 'Audi is on the verge of selling a shareholding in its Formula 1 Sauber team to Qatar. An announcement could come ahead of the Qatar Grand Prix next week. There is also said to be the possibility of an investment that is', "Max Verstappen was one of the first cars on the scene of Lance Stroll's crash, seeing his lap ruined by the double-waved yellow. Ferrari's Charles Leclerc improved his lap time to dump Verst", 'Jennie Gow suffered a stroke in December 2022, leaving her unable to read, write or speak. The BBC 5 Live radio journalist has now written a book about Formula 1. Titled How to Read F1, the book is a', 'Liam Lawson was the only driver with more than 10 grand prix starts to start the race in Brazil. Oliver Bearman was penalised for a 360° pirouette in Turn 7 when 

In [6]:
filtered = []
filteredText = ''

def filter_relevant_summary(summary, keywords):
    # Check if any keyword is in the summary
    for keyword in keywords:
        if keyword.lower() in summary.lower():
            return True
    return False  # Return None if no relevant keyword found
teams = ['Sauber', 'Alpine', 'Aston Martin', 'Aston', 'Ferrari', 'Haas', 'McLaren', 'Mercedes', 'RedBull', 'Red Bull', 'Williams',
         'Visa Cash App RB', 'VCARB']

drivers = ['bottas', 'zhou', 'guanyu', 'esteban', 'ocon', 'pierre', 'gasly', 'fernando', 'alonso', 'lance',
           'stroll', 'charles', 'leclerc', 'carlos', 'sainz', 'kevin', 'magnussen', 'nico', 'hulkenberg', 'lando', 'norris', 'oscar', 'piastri',
           'lewis', 'hamilton', 'george', 'russell', 'max', 'verstappen', 'sergio', 'perez', 'alex', 'albon', 'logan', 'sargeant', 'yuki', 'tsunoda',
           'nyck', 'de vries', 'franco', 'colapinto']

event = ['grand prix', 'pit stops', 'qualifying', 'quali', 'practice sessions', 'driver\'s briefing', 'press conference', 'race strategy',
         'safety car', 'red flag', 'yellow flag', 'track condition', 'penalties', 'penalty', 'tyre choice', 'tire choice','steward',
         'race weekend', 'weather forecasts', 'race result', 'championship standings', 'incidents', 'overtake', 'passes', 'pit lane', 'lap times',
         'reprimand', 'adjustment of', 'tech','disqualified','pit']

keywords = teams + drivers + event

for summary in summaryResult:
    if filter_relevant_summary(summary,keywords):
        filtered.append(summary)
        filteredText += summary
print(filtered)

["Brazilians flocked to their home grand prix this weekend to cheer on Williams sensation Franco Colapinto. The country's last full-time F1 driver Felipe Massa is yet to find a successor after his retirement at the", 'Audi is on the verge of selling a shareholding in its Formula 1 Sauber team to Qatar. An announcement could come ahead of the Qatar Grand Prix next week. There is also said to be the possibility of an investment that is', "Max Verstappen was one of the first cars on the scene of Lance Stroll's crash, seeing his lap ruined by the double-waved yellow. Ferrari's Charles Leclerc improved his lap time to dump Verst", 'Liam Lawson was the only driver with more than 10 grand prix starts to start the race in Brazil. Oliver Bearman was penalised for a 360° pirouette in Turn 7 when chasing Sainz on lap 36.', 'Gabriel Bortoleto looks set to join Sauber next season. Brazilian had been a frontrunner to become Nico Hulkenberg’s team-mate for 2025. Sauber wanted to take its time weighin

In [7]:
#Shorten to 280

def shorten(text):
    inputs = tokenizer(text, max_length=1024, return_tensors="pt", truncation=True)
    summary_ids = model.generate(inputs["input_ids"], max_length=270, min_length=10, length_penalty=2.0, num_beams=4, early_stopping=True)
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

short = shorten(filteredText)
print(short)

Max Verstappen won the rain-hit Brazilian Grand Prix in Sao Paulo. The Red Bull driver increased his lead in the drivers' championship to 62 points. He can clinch his fourth straight title in the Las Vegas Grand Prix.


In [14]:
# Reformat text

# Split text by period to separate sentences
sentences = short.split('.')

# Format each sentence with a bullet point
bullet_points = "\n".join([f"• {sentence.strip()}" for sentence in sentences if sentence.strip()])

print(bullet_points)
print(len(bullet_points))

• Max Verstappen won the rain-hit Brazilian Grand Prix in Sao Paulo
• The Red Bull driver increased his lead in the drivers' championship to 62 points
• He can clinch his fourth straight title in the Las Vegas Grand Prix
220


In [15]:
import requests
import json
from requests_oauthlib import OAuth1

API_KEY = 'MRcrFnNBKj5U7YKEa7Msm8Oc1'
API_SECRET_KEY = 'YMOWRJCcaWfM9CBMv2jZnFi25LJPHdUt2Eb91qesU0Xa7D1eL4'
ACCESS_TOKEN = '1855362853575782401-YDrWgNT6bDomcXfpeIOIp17JLl0R3C'
ACCESS_TOKEN_SECRET = 'afZb6osNib8DxQ7PBC8Ox3ZeZec8L6pXAQAvzynIupMAH'

# Replace with your Bearer Token
# BEARER_TOKEN = 'AAAAAAAAAAAAAAAAAAAAAESFwwEAAAAAiGAFXA0JvLG6LoH6V8V%2B26bwK3E%3D3NngaDoCGWSNYpgbjZTPQPks2fLEWdX7Tam6N7tafRkamjmfYp'

auth = OAuth1(API_KEY, API_SECRET_KEY, ACCESS_TOKEN, ACCESS_TOKEN_SECRET)

headers = {
  'content-type': 'application/json',
};

# Create the tweet content
tweet_content = {
    "text": f'{bullet_points}'
}

# API URL for posting a tweet
url = 'https://api.x.com/2/tweets'

# Make the POST request to the Twitter API
response = requests.post(url, auth=auth, json=tweet_content, headers=headers)

# Check if the tweet was posted successfully
if response.status_code == 201:
    print("Tweet posted successfully!")
else:
    print(f"Error: {response.status_code} - {response.text}")

Tweet posted successfully!
